# 第11回 顔を、少ない数で持つ（解答例・教員用）


---


## 前回からの接続


第9回は、ペンギンの4本の列を、混合正規分布で楕円の群に分けました。今日のデータは、1行に4,096本の列があります。1行が、顔の写真1枚です。たくさんの数を、できるだけ情報を失わずに、少ない数で持つ方法を考えます。


---


## 今日の分析目標


顔の画像1枚（4,096個の数）を、誰の顔か見分ける力を保ったまま、何個の数で持てるかに答える。

まず、1行の4,096個を64×64に並べ直して、顔として描きます。次に、2画素だけで、点の雲を1本の向きで近似し、誤差が最小の向きを探します。そして、4,096画素で主成分分析（principal component analysis）をかけ、向きを顔として描きます。さらに、少ない本数の向きから顔を戻し、何本で何割を持てるかを見ます。最後に、少ない数で持った顔で、誰の顔か見分けられるかを測ります。

もう1つのねらいは、`reshape`で配列の形を替えて、1行の数の並びと画像を行き来できるようになることです。縮めるのも戻すのも、行列の積`@`で書きます（第2回のおさらい）。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

研究所が1990年代に撮った顔写真が、研究用に公開されています。画像も、数を並べた表として配られます。今日は、scikit-learnが配っている版を使います。


### 場面

写真は、小さな点（画素）の明るさを並べたものです。64×64の写真なら、4,096個の数です。でも、顔どうしは似ています。4,096個も要らないのではないでしょうか。

顔の写真1枚を、もっと少ない数で持ちたい。少なくしても、誰の顔かを見分けられるくらいの情報は残したい。何個の数まで減らせるかを確かめます。


### 手元のデータ

Olivettiの顔画像（AT&T Laboratories Cambridge、1992〜1994年撮影）です。40人×10枚＝400枚で、1行が1枚、4,096画素の明るさ（0が黒、1が白）が並びます。列`person`が人の番号（0〜39）で、分けたあとの答え合わせに使います。

使うときは、"AT&T Laboratories Cambridge"のクレジットを書きます。


### 最後に出す答え

顔の写真1枚を、何個の数で持てるか。その数で、誰の顔か見分ける力が残るかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. 顔は数の表：4,096個の数を64×64に並べ直すと、顔になります。
2. 1本の向き：2画素の点の雲を、1本の直線で近似します。誤差がいちばん小さい向きを探します。
3. 顔の向き：えっ、顔どうしの違いでいちばん大きい向きは、人の違いではなく、顔全体の明るさでした。
4. 少ない数から戻す：50個の数からでも、顔がほぼ戻ります。
5. 見分ける：50個の数でも、画素そのまま（4,096個）とほぼ同じくらい見分けられます。

このあと、この5つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| 顔は数の表（1節） | 顔は数の表か？ | 1行の4,096個を64×64に並べ直して描き、平均の顔とずれを見る |
| 1本の向き（2節） | 1本の向きで近似するには？ | 2画素で向きを回して誤差を測り、`PCA`と比べる |
| 顔の向き（3節） | 向きは何を表す？ | 4,096画素で`PCA`をかけ、向きを顔として描く |
| 少ない数から戻す（4節） | 少ない数から戻るか？ | 得点から顔を戻し、本数ごとの累積寄与率を見る |
| 見分ける（5節） | 見分ける力は残るか？ | いちばん近い顔で見分け、交差検証の正解率を比べる |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- `reshape`で、1行の数の並びと画像を行き来できる
- 主成分分析が、平均からのずれを誤差が最小（散らばりが最大）の向きで近似することを、2画素の図で説明できる
- `PCA`の向きを顔として描き、両端の顔から向きの意味を読める
- 得点から顔を戻す計算を、行列の積で読み、累積寄与率（cumulative proportion of variance explained）で本数ごとの割合を読める
- 縮めた数で最近傍法を使い、交差検証（cross-validation）で見分ける力を比べられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


---


## 1. 顔は数の表か？


ここから、段1「顔は数の表」（1節）です。問いは、顔の写真は、本当に数の表として扱えるか、です。


### お手本：表を読み込む

- 役割：顔の表を読み込み、形と先頭を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`olivetti_faces.csv.gz`
- 出力：表`df`（400行×4,097列）


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/olivetti_faces.csv.gz', index_col=0)
print(df.shape)
df.iloc[:3, :5]


#### 1行ずつ読む

- `DATA_DIR`の行：データの置き場所を`DATA_DIR`にしまう（Colabでは、置き場所のURLが2行に分かれて入っている）
- `pd.read_csv`の行：圧縮した表を読み、表`df`にしまう。`index_col=0`で、先頭の列（写真の番号）を行の名前にする
- `print`の行と最後の行：形（400行×4,097列）と、先頭3行の5列を見る


### お手本：人の番号と画素に分ける

- 役割：人の番号と、4,096画素の明るさを、別々の配列にする
- 入力：表`df`
- 出力：配列`person`（400個）と`X`（400行×4,096列）


In [ ]:
person = df['person'].values
X = df.drop(columns='person').values
print(X.shape, X.min(), X.max())


#### 1行ずつ読む

- 1行目：人の番号の列を、配列`person`にする（`values`で配列に）
- 2行目：残りの4,096列を、配列`X`にする
- 3行目：形と、明るさの最小・最大を見る。0から1のあいだです


### 1行を、画像に並べ直す

`X`の1行は、4,096個の数が1列に並んだものです。写真は64行64列で、左上から右へ、1行ずつ並べてあります。1列の並びを、64×64の形に並べ直したいのです。


### reshape：最小の例

- 役割：12個の並びを、3行4列に並べ直し、また戻す
- 入力：なし
- 出力：3つの表示


In [ ]:
a = np.arange(12)
print(a)
b = a.reshape(3, 4)
print(b)
print(b.reshape(12))


#### 1行ずつ読む

- 1・2行目：0から11までの12個を、1列に並べた配列`a`を作る
- 3・4行目：`a`を、3行4列の形に並べ直して`b`にする。前から4個ずつ、行に詰める
- 5行目：`b`を、もとの12個の並びに戻す


#### 中身を見る：形


In [ ]:
print(a.shape, b.shape)
print(b[1])


形（`shape`）だけが変わりました。`b`の1行目は、`a`の4〜7番目です。並べ直しても、数の順は変わりません。


> 今日の重要概念（Python）：reshape
>
> reshapeは、配列の形を替える。並びの順はそのままで、前から順に新しい形に詰める。1行の4,096個は、64×64の画像になり、また1行に戻せる。


### お手本：1枚を画像で見る

- 役割：0枚目の4,096個を64×64に並べ直し、画像として描く
- 入力：配列`X`の0行目
- 出力：図


In [ ]:
img = X[0].reshape(64, 64)
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()


#### 1行ずつ読む

- 1行目：0枚目の4,096個を、64×64に並べ直して`img`にする
- 2行目：数の表を、明るさの画像として描く（`imshow`、`cmap='gray'`は白黒）
- 3行目：目盛りを消す
- 4行目：図を表示する


### お手本：訓練とテストに分ける

最後の段で、見分ける力をテストで1回だけ測ります。そのために先に分けておきます（第4回）。

- 役割：400枚を、訓練300枚とテスト100枚に分ける
- 入力：配列`X`と`person`
- 出力：`X_tr`・`X_te`・`p_tr`・`p_te`


In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_te, p_tr, p_te = train_test_split(
    X, person, test_size=0.25, random_state=0, stratify=person)
print(X_tr.shape, X_te.shape)


#### 1行ずつ読む

- 1行目：分ける道具を呼び出す
- 2・3行目：4分の1をテストにする。`stratify=person`で、どの人も訓練とテストに同じ割合で入る
- 4行目：訓練300枚、テスト100枚

このあとの計算は、すべて訓練の300枚で行います。


### お手本：平均の顔

- 役割：訓練の300枚の、画素ごとの平均を出して描く
- 入力：配列`X_tr`
- 出力：`mean_face`（4,096個）と図


In [ ]:
mean_face = X_tr.mean(axis=0)
plt.imshow(mean_face.reshape(64, 64), cmap='gray')
plt.axis('off')
plt.show()


#### 1行ずつ読む

- 1行目：300枚の、画素ごとの平均を出す（`axis=0`で縦に）。4,096個の数
- 2行目：その4,096個を64×64に並べ直して描く
- 3・4行目：目盛りを消して表示する

ぼんやりした顔です。300枚に共通の形です。


In [ ]:
# @title 図：1枚目、平均の顔、平均からのずれ（中身は読み飛ばしてよい）
dev = X_tr[0] - mean_face
v = np.abs(dev).max()
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 3, 1)
plt.imshow(X_tr[0].reshape(64, 64), cmap='gray')
plt.title('1枚目')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(mean_face.reshape(64, 64), cmap='gray')
plt.title('平均の顔')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(dev.reshape(64, 64), cmap='bwr', vmin=-v, vmax=v)
plt.title('ずれ（1枚目−平均）')
plt.axis('off')
plt.show()


赤は平均より明るい画素、青は暗い画素です。どの顔も、平均の顔＋ずれです。

$i$枚目の4,096個の並びを$\mathbf{x}_i$、平均の顔を$\bar{\mathbf{x}}$と書きます。

$$
\bar{\mathbf{x}} = \frac{1}{n} \sum_{i=1}^{n} \mathbf{x}_i, \qquad \mathbf{x}_i = \bar{\mathbf{x}} + (\mathbf{x}_i - \bar{\mathbf{x}})
$$

平均は、どの顔にも共通です。ずれを少ない数で持てれば、顔を少ない数で持てます。


### 結果を見る前に、予想を書く

ずれの4,096個のうち、本当に要るのは何個くらいでしょうか。予想と、その理由を1行で書いてください。


（ここに予想を書く）


### TODO①：形を替える（予想 → 実行 → 比べる）

下のセルは、0枚目の4,096個を64×64に並べ直して描きます。

まず、予想を書いてください。同じ4,096個を、32行128列に並べ直すと、どんな画像になるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`64, 64`を`32, 128`に替えて実行してください。


In [ ]:
# TODO: 64, 64 を 32, 128 に替えて実行する

img = X[0].reshape(32, 128)
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。どんな画像になりましたか。なぜでしょうか。


（ここに答えを書く）


細い顔が2つ並びます。もとの画像の2行ぶんが、新しい1行になったからです。左にもとの偶数の行、右に奇数の行が並びます。並びの順は変わらず、詰め方だけが変わりました。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 顔は数の表（1節） | 1枚は4,096個の数。64×64に並べ直すと顔になる |
| 1本の向き（2節） | ？（このあとの段で） |
| 顔の向き（3節） | ？（このあとの段で） |
| 少ない数から戻す（4節） | ？（このあとの段で） |
| 見分ける（5節） | ？（このあとの段で） |

まだ分からないのは、ずれを、少ない数でどう持つか、です。


---


## 2. 1本の向きで近似するには？


ここから、段2「1本の向き」（2節）です。問いは、平均からのずれを、1本の向きで近似するには、どの向きを選べばよいか、です。

1行の4,096個は、64×64に並べ直すと顔になりました。どの顔も、平均の顔＋ずれです。4,096画素では図にできないので、まず2画素で考えます。


### お手本：2画素を取り出す

- 役割：300枚の、左右の目のあたりの2画素だけを取り出す
- 入力：配列`X_tr`
- 出力：配列`two`（300行2列）と、2画素の相関係数


In [ ]:
two = X_tr[:, [1300, 1324]]
print(two.shape)
print(np.corrcoef(two[:, 0], two[:, 1])[0, 1].round(3))


#### 1行ずつ読む

- 1行目：300枚の、1300番目と1324番目の画素だけを取り出す。20行目の20列目は$20 \times 64 + 20 = 1300$番目、44列目は1324番目
- 2行目：形は300行2列
- 3行目：2画素の相関係数を見る

左右の目のあたりです。片方が明るい顔は、もう片方も明るくなっています。


### お手本：ずれの点の雲

- 役割：2画素の平均を引いてずれにし、1枚を1つの点として打つ
- 入力：配列`two`
- 出力：配列`xc`（300行2列）と図


In [ ]:
xc = two - two.mean(axis=0)
plt.scatter(xc[:, 0], xc[:, 1], s=10)
plt.xlabel('左の目のあたり（平均からのずれ）')
plt.ylabel('右の目のあたり')
plt.axis('equal')
plt.show()


#### 1行ずつ読む

- 1行目：2画素それぞれの平均を引き、平均からのずれ`xc`にする
- 2行目：1枚を1つの点として打つ
- 3・4行目：軸の名前を付ける
- 5行目：縦と横の目盛りの幅をそろえる

点の雲は、右上がりの細長い形です。


### 1本の向きで近似する

原点を通る直線を1本引き、それぞれの点を、直線の上のいちばん近い点で近似します。近似した点は、直線の上の位置（得点）1個で表せます。2個の数が、1個になります。誤差は、点から直線への垂線の長さです。

向き（長さ1）を$\mathbf{w} = (\cos\theta, \sin\theta)$、点を$\mathbf{x}_i$とします。

$$
t_i = \mathbf{x}_i \cdot \mathbf{w}, \qquad \hat{\mathbf{x}}_i = t_i \mathbf{w}, \qquad
E = \sum_{i=1}^{n} \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2
$$

$t_i$が得点、$\hat{\mathbf{x}}_i$が近似、$E$が誤差（距離の二乗の和）です。


### お手本：30度の向きの誤差

- 役割：30度の向きで、得点と近似を出し、誤差を測る
- 入力：配列`xc`
- 出力：得点`t`（300行1列）と誤差`err`


In [ ]:
theta = np.deg2rad(30)
W = np.array([[np.cos(theta), np.sin(theta)]])
t = xc @ W.T
err = ((xc - t @ W) ** 2).sum()
print(t.shape, err.round(2))


#### 1行ずつ読む

- 1行目：30度を、計算用の単位（ラジアン）にする
- 2行目：向き`W`（1行2列）を作る（`np.cos`・`np.sin`は余弦と正弦）
- 3行目：得点。ずれ（300行2列）`@`向きの転置（2行1列）で、300行1列（第2回の`@`と`.T`）
- 4行目：近似は得点`@`向き。ずれとの差を2乗して、全部足す
- 5行目：得点の形と誤差を表示する


#### 中身を見る：先頭の3枚


In [ ]:
print(xc[:3].round(3))
print(t[:3].round(3))
print((t @ W)[:3].round(3))


上からずれ（2個）、得点（1個）、近似（2個）です。2個の数を、得点1個で近似しました。


### お手本：角度を回す

- 役割：角度を0度から179度まで1度ずつ替えて誤差を測り、いちばん小さい角度を探す
- 入力：配列`xc`
- 出力：リスト`errs`（180個）と、いちばん小さい角度`best`


In [ ]:
errs = []
for deg in range(180):
    theta = np.deg2rad(deg)
    W = np.array([[np.cos(theta), np.sin(theta)]])
    t = xc @ W.T
    errs.append(((xc - t @ W) ** 2).sum())
best = np.argmin(errs)
print(best, round(errs[best], 2))


#### 1行ずつ読む

- 1・2行目：誤差をためるリストを作り、角度を0度から179度まで1度ずつ替える
- 3〜6行目：お手本と同じ計算で誤差を出し、リストの後ろに足す
- 7行目：いちばん小さい誤差の位置（`argmin`）。位置がそのまま角度
- 8行目：その角度と誤差を表示する


In [ ]:
plt.plot(range(180), errs)
plt.xlabel('向きの角度（度）')
plt.ylabel('距離の二乗の和')
plt.show()


45度で誤差がいちばん小さくなります。点の雲の、細長い向きです。


### お手本：PCAと比べる

- 役割：主成分分析の道具`PCA`で、向きを1本だけ求め、45度の向きと比べる
- 入力：配列`two`
- 出力：`PCA`の向きと、45度の向き


In [ ]:
from sklearn.decomposition import PCA
pca2 = PCA(n_components=1)
pca2.fit(two)
print(pca2.components_.round(3))
print(np.round([np.cos(np.deg2rad(45)), np.sin(np.deg2rad(45))], 3))


#### 1行ずつ読む

- 1〜3行目：主成分分析の道具`PCA`を呼び出し、向き1本で2画素に当てはめる
- 4行目：`PCA`が選んだ向き（第1主成分、`components_`）
- 5行目：45度の向き

ほぼ同じです（1度きざみで回したぶんだけ違います）。


### PCAがしていること

- 平均を引いて、ずれにする（自分で`xc`を作らなくてよい）
- 誤差が最小になる向きを探す。角度を回す代わりに、行列の計算（特異値分解）で一度に求める
- 2本目は、1本目と直角な向きのうちで、残りの誤差が最小の向き


### 誤差と散らばりは、足すと一定

点、近似、誤差は、直角三角形を作ります（三平方の定理）。

$$
\|\mathbf{x}_i\|^2 = t_i^2 + \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2
$$

全部の点で足すと、左辺は向きによらず一定です。だから、誤差が最小の向きは、得点の散らばり（得点の二乗和）が最大の向きです。


In [ ]:
# @title 図：向きごとの誤差と、得点の二乗和（中身は読み飛ばしてよい）
sc2 = []
for deg in range(180):
    theta = np.deg2rad(deg)
    W = np.array([[np.cos(theta), np.sin(theta)]])
    sc2.append(((xc @ W.T) ** 2).sum())
plt.plot(range(180), errs, label='誤差')
plt.plot(range(180), sc2, label='得点の二乗和（散らばり）')
plt.xlabel('向きの角度（度）')
plt.ylabel('距離の二乗の和')
plt.legend()
plt.show()


> 今日の重要概念（統計）：主成分分析
>
> 主成分分析は、平均からのずれを、誤差が最小の向きで近似する。誤差が最小の向きは、散らばりが最大の向きでもある。


### TODO②：直角の向きにする（予想 → 実行 → 比べる）

下のセルは、1つの向きで、誤差（左）と得点の二乗和（右）を表示します。

まず、予想を書いてください。45度と直角な135度では、2つの数はどうなるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`45`を`135`に替えて実行してください。


In [ ]:
# TODO: 45 を 135 に替えて実行する

theta = np.deg2rad(135)
W = np.array([[np.cos(theta), np.sin(theta)]])
t = xc @ W.T
print(((xc - t @ W) ** 2).sum().round(2), (t ** 2).sum().round(2))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。2つの数は、45度のときと比べてどうなりましたか。


（ここに答えを書く）


135度では、誤差と得点の二乗和が入れ替わります（45度は0.67と6.6、135度は6.6と0.67）。誤差は最大、散らばりは最小です。2つの和は、どちらの向きでも同じです。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 顔は数の表（1節） | 1枚は4,096個の数。64×64に並べ直すと顔になる |
| 1本の向き（2節） | 2画素では、誤差が最小の向きは45度。散らばりが最大の向きと同じ |
| 顔の向き（3節） | ？（このあとの段で） |
| 少ない数から戻す（4節） | ？（このあとの段で） |
| 見分ける（5節） | ？（このあとの段で） |

まだ分からないのは、4,096画素の顔では、どんな向きが選ばれるか、です。


---


## 3. 向きは何を表す？


ここから、段3「顔の向き」（3節）です。問いは、4,096画素の顔で、誤差が最小の向きは、何を表すか、です。

2画素では、誤差が最小の向きは45度で、散らばりが最大の向きでもありました。`PCA`は、その向きを、行列の計算で一度に求めます。ここから、4,096画素すべてで向きを求めます。


### お手本：顔でPCA

- 役割：訓練の300枚で、本数を決めずに主成分分析をかける
- 入力：配列`X_tr`（人の番号は使わない）
- 出力：`pca`（向きと寄与率）


In [ ]:
pca = PCA(random_state=0)
pca.fit(X_tr)
print(pca.components_.shape)
print(pca.explained_variance_ratio_[:3].round(3))


#### 1行ずつ読む

- 1・2行目：本数を決めずに、訓練の300枚に当てはめる。人の番号は使わない
- 3行目：向きは300本。1本が4,096個の数（向きは、枚数より多くは作れない）
- 4行目：1本ずつの寄与率（散らばりのうち、その向きが持つ割合）

第1の向きだけで、ずれの散らばりの0.23（23%）を持ちます。第2は0.141、第3は0.081です。散らばりの大きい順に、向きが並んでいます。


### お手本：向きを顔として描く

- 役割：上の3本の向きを、64×64に並べ直して描く
- 入力：`pca.components_`
- 出力：図


In [ ]:
for k in range(3):
    plt.subplot(1, 3, k + 1)
    plt.imshow(pca.components_[k].reshape(64, 64), cmap='bwr')
    plt.axis('off')
plt.show()


#### 1行ずつ読む

- 1・2行目：向きを3本、1行3列の図に並べる
- 3行目：1本の向き（4,096個）を64×64に並べ直して描く。赤は正、青は負
- 4・5行目：目盛りを消して表示する

向きも4,096個の数なので、顔と同じ形の画像になります（固有顔）。


### お手本：特異値分解と比べる

- 役割：平均を引いた表を特異値分解（singular value decomposition）し、1本目の向きが`PCA`と同じか確かめる
- 入力：配列`X_tr`と`mean_face`
- 出力：2本の向きの内積の大きさ


In [ ]:
U, S, Vt = np.linalg.svd(X_tr - mean_face, full_matrices=False)
print(np.abs(Vt[0] @ pca.components_[0]).round(3))


#### 1行ずつ読む

- 1行目：ずれの表（300行4,096列）を特異値分解する。`Vt`の行が向き
- 2行目：2本の向き（どちらも長さ1）の内積の大きさ。1なら同じ向き（符号は逆でもよい）


### お手本：得点を出す

第1の向きの得点が、いちばん小さい顔と、いちばん大きい顔を並べると、その向きに沿って何が変わっているかが見えます。得点は、ずれ`@`向きの転置で、300枚それぞれに出ます。


In [ ]:
t_all = pca.transform(X_tr)
print(t_all.shape)
print(t_all[:3, :2].round(2))


#### 1行ずつ読む

- 1行目：訓練の300枚の得点（`transform`）
- 2行目：300枚×300本
- 3行目：先頭3枚の、第1・第2の得点


In [ ]:
# @title 図：第1の向きの両端の顔（中身は読み飛ばしてよい）
order = np.argsort(t_all[:, 0])
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 3, 1)
plt.imshow(X_tr[order[0]].reshape(64, 64), cmap='gray')
plt.title('得点がいちばん小さい顔')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(pca.components_[0].reshape(64, 64), cmap='bwr')
plt.title('第1の向き')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(X_tr[order[-1]].reshape(64, 64), cmap='gray')
plt.title('得点がいちばん大きい顔')
plt.axis('off')
plt.show()


えっ、いちばん小さい顔は暗く、いちばん大きい顔は明るい顔です。人の違いではありません。数で確かめます。


### お手本：明るさと比べる


In [ ]:
light = X_tr.mean(axis=1)
print(np.corrcoef(t_all[:, 0], light)[0, 1].round(3))


#### 1行ずつ読む

- 1行目：1枚ごとに、4,096画素の平均（顔全体の明るさ）を出す（`axis=1`で横に）
- 2行目：第1の得点との相関係数

0.981です。顔どうしの違いでいちばん大きいのは、撮ったときの光の量でした。


In [ ]:
plt.scatter(t_all[:, 0], light, s=10)
plt.xlabel('第1の向きの得点')
plt.ylabel('顔全体の明るさの平均')
plt.show()


> 重要：主成分分析は、人の番号を使わずに、散らばりの大きい向きを探す。だから、いちばん大きい向きが、知りたい違いを表すとは限らない。


### TODO③：第2の向き（予想 → 実行 → 比べる）

下のセルは、第`k`の向きの得点がいちばん小さい顔、向き、いちばん大きい顔を並べます。`np.argsort`は、小さい順に並べたときの番号です。

まず、予想を書いてください。第2の向き（`k = 1`）では、両端の顔は何が違うでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`0`を`1`に替えて実行してください。


In [ ]:
# TODO: 0 を 1 に替えて実行する

k = 1
order = np.argsort(t_all[:, k])
plt.subplot(1, 3, 1)
plt.imshow(X_tr[order[0]].reshape(64, 64), cmap='gray')
plt.subplot(1, 3, 2)
plt.imshow(pca.components_[k].reshape(64, 64), cmap='bwr')
plt.subplot(1, 3, 3)
plt.imshow(X_tr[order[-1]].reshape(64, 64), cmap='gray')
plt.show()


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。両端の顔は、何が違いましたか。


（ここに答えを書く）


第2の向きは、光が左から当たるか、右から当たるかです。下のセルで、顔の左半分と右半分の明るさの差と、第2の得点の相関が0.935になることを確かめられます。


In [ ]:
faces = X_tr.reshape(300, 64, 64)
lr = faces[:, :, :32].mean(axis=(1, 2)) - faces[:, :, 32:].mean(axis=(1, 2))
print(np.corrcoef(t_all[:, 1], lr)[0, 1].round(3))


`X_tr`を300枚×64×64の形にし（`reshape`）、左の32列と右の32列の明るさの平均の差を出しました。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 顔は数の表（1節） | 1枚は4,096個の数。64×64に並べ直すと顔になる |
| 1本の向き（2節） | 2画素では、誤差が最小の向きは45度。散らばりが最大の向きと同じ |
| 顔の向き（3節） | 第1の向きは顔全体の明るさ（相関0.981）、第2は左右の光 |
| 少ない数から戻す（4節） | ？（このあとの段で） |
| 見分ける（5節） | ？（このあとの段で） |

まだ分からないのは、少ない本数の向きから、顔が戻るか、です。


---


## 4. 少ない数から戻るか？


ここから、段4「少ない数から戻す」（4節）です。問いは、少ない本数の向きの得点から、顔は戻るか、です。

4,096画素で`PCA`をかけると、向きは散らばりの大きい順に並びます。第1の向きは顔全体の明るさ、第2は左右の光でした。ここから、上の何本かだけで、顔を持ちます。


### 縮めて、戻す

- 縮める：ずれを、上の$k$本の向きの得点$k$個にする
- 戻す：得点に向きを掛けて足し、平均の顔を足す

どちらも、行列の積`@`で書けます。上の$k$本の向きを並べた行列を$W$（$k$行4,096列）とします。

$$
\mathbf{t} = (\mathbf{x} - \bar{\mathbf{x}}) W^\top, \qquad
\hat{\mathbf{x}} = \mathbf{t} W + \bar{\mathbf{x}}
$$

$\mathbf{t}$は$k$個の得点です。4,096個を$k$個にしました。


### お手本：50本で戻す

- 役割：テストの0枚目を、50本の向きの得点に縮めて、また戻す
- 入力：`pca.components_`、`X_te[0]`、`mean_face`
- 出力：得点`t`（50個）と、戻した顔`x_hat`（4,096個）


In [ ]:
W50 = pca.components_[:50]
t = (X_te[0] - mean_face) @ W50.T
x_hat = t @ W50 + mean_face
print(W50.shape, t.shape, x_hat.shape)


#### 1行ずつ読む

- 1行目：上の50本の向き（50行4,096列）を取り出す
- 2行目：テストの0枚目のずれを、50個の得点に縮める
- 3行目：得点に向きを掛けて、平均の顔を足す。4,096個に戻る
- 4行目：3つの形を表示する


#### 中身を見る：50個の得点


In [ ]:
print(t[:5].round(2))
print(X_te[0][:5].round(2), x_hat[:5].round(2))


上は50個の得点の先頭5個、下はもとの画素の先頭5個と、戻した画素の先頭5個です。


In [ ]:
plt.subplot(1, 2, 1)
plt.imshow(X_te[0].reshape(64, 64), cmap='gray')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(x_hat.reshape(64, 64), cmap='gray')
plt.axis('off')
plt.show()


4,096個のかわりに、50個の得点と、みんなで使う向き50本と平均の顔で持てます。


### お手本：本数を替えて戻す

- 役割：本数を5・20・50・100本に替えて、テストの0枚目を戻して並べる
- 入力：`pca.components_`、`X_te[0]`、`mean_face`
- 出力：図


In [ ]:
ks = [5, 20, 50, 100]
for j in range(4):
    W_k = pca.components_[:ks[j]]
    x_k = (X_te[0] - mean_face) @ W_k.T @ W_k + mean_face
    plt.subplot(1, 4, j + 1)
    plt.imshow(x_k.reshape(64, 64), cmap='gray')
    plt.title(ks[j])
    plt.axis('off')
plt.show()


#### 1行ずつ読む

- 1・2行目：本数を4通り並べ、1つずつ取る
- 3行目：上の`ks[j]`本の向きを取り出す
- 4行目：縮めて戻すを1行で書いた（ずれ`@`向きの転置`@`向き＋平均）
- 5〜8行目：1行4列の図の`j + 1`番目に描き、本数を題にする

5本では、誰の顔か分かりません。明るさと光の向きくらいです。20本で顔の形が出て、50本でほぼもとの顔です。では、何本あれば十分でしょうか。


### お手本：累積寄与率

- 役割：寄与率（proportion of variance explained）を前から足し上げ、本数ごとに持てる割合を出す
- 入力：`pca.explained_variance_ratio_`
- 出力：`cum`（300個）と、初めて0.9を超える本数


In [ ]:
cum = pca.explained_variance_ratio_.cumsum()
print(cum[[9, 49, 99]].round(3))
print(np.argmax(cum > 0.9) + 1)


#### 1行ずつ読む

- 1行目：寄与率を、前から足し上げる（`cumsum`）。$k$本までで持てる割合
- 2行目：10本・50本・100本までの累積寄与率（番号は0から数える）
- 3行目：初めて0.9を超える本数。`argmax`はTrueの最初の位置。1を足して本数にする


In [ ]:
plt.plot(range(1, 301), cum)
plt.axhline(0.9, color='gray', linestyle='--')
plt.xlabel('残す向きの本数')
plt.ylabel('累積寄与率')
plt.show()


10本で0.655、50本で0.88、61本で初めて0.9を超えます。


> 重要：累積寄与率は、上の何本の向きで、ずれの散らばりの何割を持てるかを表す。何割あれば十分かは、何に使うかで決める。


### TODO④：戻す本数を替える（予想 → 実行 → 比べる）

下のセルは、テストの1枚目を50本で戻し、もとの顔と並べて、1画素あたりの誤差の2乗の平均を表示します。

まず、予想を書いてください。10本にすると、誤差は何倍になるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`50`を`10`に替えて実行してください。


In [ ]:
# TODO: 50 を 10 に替えて実行する

W_k = pca.components_[:10]
x_k = (X_te[1] - mean_face) @ W_k.T @ W_k + mean_face
plt.subplot(1, 2, 1)
plt.imshow(X_te[1].reshape(64, 64), cmap='gray')
plt.subplot(1, 2, 2)
plt.imshow(x_k.reshape(64, 64), cmap='gray')
plt.show()
print(((X_te[1] - x_k) ** 2).mean().round(4))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。誤差は何倍になりましたか。顔はどう変わりましたか。


（ここに答えを書く）


下のセルで、2つの本数の誤差を並べて確かめられます。10本では、誤差の2乗の平均が0.0055から0.0071へ、約1.3倍になります。数は少し増えただけでも、顔の細かいところが消えます。


In [ ]:
for k in [50, 10]:
    W_k = pca.components_[:k]
    x_k = (X_te[1] - mean_face) @ W_k.T @ W_k + mean_face
    print(k, ((X_te[1] - x_k) ** 2).mean().round(4))


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 顔は数の表（1節） | 1枚は4,096個の数。64×64に並べ直すと顔になる |
| 1本の向き（2節） | 2画素では、誤差が最小の向きは45度。散らばりが最大の向きと同じ |
| 顔の向き（3節） | 第1の向きは顔全体の明るさ（相関0.981）、第2は左右の光 |
| 少ない数から戻す（4節） | 50本で累積寄与率0.88。50個の数から顔が戻る |
| 見分ける（5節） | ？（このあとの段で） |

まだ分からないのは、少ない数で持った顔で、誰の顔か見分けられるか、です。


---


## 5. 見分ける力は残るか？


ここから、段5「見分ける」（5節）です。問いは、少ない数で持った顔で、誰の顔かを見分ける力は残るか、です。

50個の得点から、顔はほぼ戻りました。累積寄与率は88%です。残りの12%を捨てて、見分ける力は落ちないでしょうか。見分ける力を物差しにして確かめます。


### いちばん近い顔で見分ける

テストの顔に、いちばん近い訓練の顔を探し、その顔の人の番号を答えにします（1-最近傍法）。近さは、画素ごとの差を2乗して足したもの（距離の二乗）です。


### お手本：いちばん近い顔

- 役割：テストの0枚目に、いちばん近い訓練の顔を探す
- 入力：`X_tr`、`X_te[0]`
- 出力：近い顔の番号、その人の番号、本当の人の番号


In [ ]:
d = ((X_tr - X_te[0]) ** 2).sum(axis=1)
near = np.argmin(d)
print(near, p_tr[near], p_te[0])


#### 1行ずつ読む

- 1行目：訓練の300枚それぞれから、テストの0枚目を引き、2乗して横に足す。300個の距離
- 2行目：いちばん近い訓練の顔の番号
- 3行目：その顔の人の番号と、本当の人の番号


In [ ]:
# @title 図：テストの0枚目と、いちばん近い訓練の顔（中身は読み飛ばしてよい）
plt.subplot(1, 2, 1)
plt.imshow(X_te[0].reshape(64, 64), cmap='gray')
plt.title('テストの0枚目')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(X_tr[near].reshape(64, 64), cmap='gray')
plt.title('いちばん近い訓練の顔')
plt.axis('off')
plt.show()


同じ36番の人です。表情と向きが少し違う写真が選ばれました。


### お手本：道具で見分ける

- 役割：最近傍法の道具で、訓練の顔を覚え、テストの0枚目を見分ける
- 入力：`X_tr`・`p_tr`、`X_te[:1]`
- 出力：見分けた人の番号


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_tr, p_tr)
print(knn.predict(X_te[:1]))


#### 1行ずつ読む

- 1・2行目：最近傍法の道具を、いちばん近い1枚で決める設定で用意する
- 3行目：訓練の顔と人の番号を覚える（`fit`）
- 4行目：テストの0枚目を見分ける。上の計算と同じ36番


### KNeighborsClassifierがしていること

- `fit`では、訓練の顔と番号を覚えるだけ
- `predict`では、上のお手本と同じく、いちばん近い訓練の顔を探す
- その顔の番号を返す


### お手本：画素そのままで測る

- 役割：訓練の300枚で、5組の交差検証の正解率を出す
- 入力：`knn`、`X_tr`・`p_tr`
- 出力：正解率の平均


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(5, shuffle=True, random_state=0)
sc = cross_val_score(knn, X_tr, p_tr, cv=skf)
print(sc.mean().round(3))


#### 1行ずつ読む

- 1行目：交差検証の道具を呼び出す
- 2行目：訓練の300枚を、人の割合をそろえて5組に分ける（第4回の交差検証）
- 3行目：組ごとに、4組で覚えて残り1組を見分け、正解率を出す
- 4行目：5つの正解率の平均

4,096個の数そのままで、0.907です。


### お手本：50本で測る

4,096個を`PCA`で50個にしてから、いちばん近い顔を探します。`PCA`の向きも、覚える組だけで求めないと、測る組の情報がもれます。そこで、`PCA`と見分ける道具を、パイプライン（pipeline）でつなぎます（第7回）。


In [ ]:
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(PCA(50, random_state=0),
                     KNeighborsClassifier(n_neighbors=1))
sc = cross_val_score(pipe, X_tr, p_tr, cv=skf)
print(sc.mean().round(3))


#### 1行ずつ読む

- 1行目：パイプラインの道具を呼び出す
- 2・3行目：50本の`PCA` → 最近傍法の順につなぐ
- 4行目：組ごとに、覚える組で向きを求め、測る組を縮めてから見分ける
- 5行目：正解率の平均

0.900です。画素そのまま（0.907）とほぼ同じです。


### お手本：テストで1回だけ

50本に決めてから、テストの100枚で1回だけ測ります。


In [ ]:
knn.fit(X_tr, p_tr)
pipe.fit(X_tr, p_tr)
print(knn.score(X_te, p_te), pipe.score(X_te, p_te))


#### 1行ずつ読む

- 1・2行目：訓練の300枚すべてで覚え直す
- 3行目：テストの100枚の正解率（`score`）。画素そのままと、50本

どちらも0.93です。


> 重要：顔どうしは似ているので、ずれの大部分は少ない向きで持てる。4,096個を50個にしても、見分ける力はほとんど落ちない。


### TODO⑤：本数を10にする（予想 → 実行 → 比べる）

下のセルは、`PCA`の本数を決めてパイプラインを作り、交差検証の正解率を出します。

まず、予想を書いてください。10個の数にすると、交差検証の正解率はどこまで下がるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`50`を`10`に替えて実行してください。


In [ ]:
# TODO: 50 を 10 に替えて実行する

pipe_x = make_pipeline(PCA(10, random_state=0),
                       KNeighborsClassifier(n_neighbors=1))
print(cross_val_score(pipe_x, X_tr, p_tr, cv=skf).mean().round(3))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。正解率は、どこまで下がりましたか。


（ここに答えを書く）


10本では0.803です。累積寄与率は0.655で、見分ける力もはっきり落ちます。下のセルで、20本なら0.877になることを確かめられます。50本は0.900でした。


In [ ]:
pipe_20 = make_pipeline(PCA(20, random_state=0),
                        KNeighborsClassifier(n_neighbors=1))
print(cross_val_score(pipe_20, X_tr, p_tr, cv=skf).mean().round(3))


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 顔は数の表（1節） | 1枚は4,096個の数。64×64に並べ直すと顔になる |
| 1本の向き（2節） | 2画素では、誤差が最小の向きは45度。散らばりが最大の向きと同じ |
| 顔の向き（3節） | 第1の向きは顔全体の明るさ（相関0.981）、第2は左右の光 |
| 少ない数から戻す（4節） | 50本で累積寄与率0.88。50個の数から顔が戻る |
| 見分ける（5節） | 交差検証の正解率：画素そのまま0.907、50本0.900 |

5つの段がそろい、今日の問いに答えられます。


### 答え

顔の画像1枚の4,096個は、50個の得点で持てます（累積寄与率0.88）。50個でも、見分ける力はほとんど落ちません（交差検証0.900、テスト0.93）。いちばん大きい向きは、人の違いではなく、明るさと光の向きでした。


まだ言えないこと

- 何本が正しいか。使い方（見分けるか、きれいに戻すか）で変わる
- 向きの意味の読み方。今日は両端の顔を見ただけ
- 光の当たり方や写り方が大きく違う写真でも、同じように言えるか


### レポートなら、こう書く

今日の解析を、期末レポートの項目に写すと、次のようになります。

- 予想：顔どうしは似ているので、4,096個も要らないだろう
- 解析：訓練の300枚で主成分分析をかけ、上の向きの得点で顔を持った
- 驚き：第1の向きは顔全体の明るさ（相関0.981）、第2は左右の光だった
- 本数：累積寄与率は50本で0.88。50本で顔がほぼ戻る
- 確かめ：最近傍法の交差検証で、画素そのまま0.907、50本0.900。テストではどちらも0.93
- 限界：本数は使い方で決めた。写り方の違う写真では確かめていない


---


## 目標に答えられたか


- 今日の目標は「顔の画像1枚を、見分ける力を保ったまま、何個の数で持てるか」でした。戻した顔と、累積寄与率と、正解率で答えられますか？
- 1節で、`reshape(64, 64)`は、4,096個をどの順に詰めましたか？
- 2節で、誤差が最小の向きが、散らばりが最大の向きと同じになるのはなぜですか？
- 3節で、第1の向きが人の違いを表さなかったのはなぜですか？
- 4節で、縮める計算と戻す計算は、それぞれどんな行列の積でしたか？
- 5節で、`PCA`をパイプラインに入れたのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：主成分分析は、平均からのずれを、誤差が最小（散らばりが最大）の向きで近似する。顔の4,096個の数は、50個の得点にしても、見分ける力をほとんど失わない
- 重要概念（統計）：主成分分析は、平均からのずれを、誤差が最小の向きで近似する。誤差が最小の向きは、散らばりが最大の向きでもある
- 重要概念（Python）：reshape。配列の形を替える。並びの順はそのままで、前から順に新しい形に詰める
- 縮める（ずれ`@`向きの転置）も、戻す（得点`@`向き＋平均）も、行列の積
- いちばん大きい向きが、知りたい違いを表すとは限らない


---


## 次回へ


今日は、向きの意味を、両端の顔を見て読みました。では、列に名前のあるデータでは、向きの意味をどう読めばよいでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：主成分分析は、平均からのずれを、誤差が最小の向きで近似する。誤差が最小の向きは、散らばりが最大の向きでもある（2節）
- Python：reshape。配列の形を替える。並びの順はそのままで、前から順に新しい形に詰める（1節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

主成分分析なら、たとえば「細長いパンを写真に撮る」を描けます。
細長いパンを、真上・真横・正面から写真に撮ります。形がいちばんよく分かるのは、影がいちばん長くなる向き（散らばりが最大）です。その向きで撮れば、写真1枚で失うもの（誤差）がいちばん少なくなります。2節の、誤差と得点の二乗和の曲線が手本になります。

</details>


<details><summary>詰まったら</summary>

reshapeなら、たとえば「卵をパックに詰める」を描けます。
1列に並んだ12個の卵を、3×4のパックに詰めます。前から順に、1段目に4個、2段目に4個、3段目に4個です。パックから出して並べ直せば、もとの1列に戻ります。1節の、12個を3行4列にする例が手本になります。

</details>


---


## 発展（任意）


### 向きに沿って、顔を動かす

平均の顔に、向きを少しずつ足したり引いたりすると、その向きの意味が見えます。動かす量は、得点の標準偏差の何倍か、で決めます。第1の向きで試します。


In [ ]:
sd = t_all[:, 0].std()
for j in range(5):
    face = mean_face + (j - 2) * sd * pca.components_[0]
    plt.subplot(1, 5, j + 1)
    plt.imshow(face.reshape(64, 64), cmap='gray', vmin=0, vmax=1)
    plt.axis('off')
plt.show()


#### 1行ずつ読む

- 1行目：第1の得点の標準偏差
- 2・3行目：−2倍から＋2倍まで5通り、平均の顔に向きを足す
- 4行目：1行5列の図の`j + 1`番目に描く
- 5行目：`vmin`・`vmax`で、明るさの0と1を黒と白にそろえる
- 6・7行目：目盛りを消して表示する

顔全体が、暗くなったり明るくなったりします。

試すなら、`components_[0]`の`0`を`1`や`2`に替えてみましょう（1行目の`[:, 0]`も同じ番号に）。第2の向きでは、光が左右に動きます。
